Original MVTec test images → deliberately change their appearance → split them → use 20% to adapt RoDA → use 80% to test RoDA.

1. What is an artificial corruption?
An artificial corruption is a change we deliberately make to an image to simulate a real-world change in environment.

In [9]:
# loading bottle dataset which is error free
source_data = "./Defect_free"

In [2]:
# loading test data = defect+normal
test_data = "./data/bottle/test"

In [4]:
import random
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from PIL import Image

In [3]:
image_size = 256
seed = 42

In [5]:

adaptation_ratio = 0.20

random.seed(seed)
torch.manual_seed(seed)

In [6]:
# transform

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [11]:
source_dataset = datasets.ImageFolder(
    root=source_data, transform=transform)

print("source data")
print("Number of source images", len(source_dataset))

source data
Number of source images 1144


In [12]:
from collections import Counter

print(Counter(source_dataset.targets))
print(source_dataset.class_to_idx)

Counter({2: 391, 3: 320, 1: 224, 0: 209})
{'bottle': 0, 'cable': 1, 'hezelnut': 2, 'screw': 3}


First we will be only testing with bottle test dataset


In [13]:
raw_test_dataset = datasets.ImageFolder(
    root=test_data,
    transform=transforms.ToTensor()
)

print("\nTARGET DATA")
print("Number of test images:", len(raw_test_dataset))


TARGET DATA
Number of test images: 83


In [16]:
# artificial corruption - reducing brightness by 30%

class BrightnessCorruption(Dataset):

    def __init__(self, dataset, brightness_factor=0.7):
        self.dataset = dataset
        self.brightness_factor = brightness_factor

    def __len__(self):
        return len(self.dataset)


    def __getitem__(self, index):

        image, label = self.dataset[index]
        image = image * self.brightness_factor
        image = torch.clamp(image, 0, 1)

        return image, label

corrupted_dataset = BrightnessCorruption(
    raw_test_dataset,
    brightness_factor=0.7
)


In [17]:
# Split 20% 80%

total_size = len(corrupted_dataset)
adaptation_size=int(
    total_size * adaptation_ratio
)

evaluation_size = total_size - adaptation_size

generator = torch.Generator().manual_seed(seed)

adaptation_dataset, evaluation_dataset = random_split(
    corrupted_dataset,
    [adaptation_size, evaluation_size],
    generator=generator
)

In [18]:
class NormalizeDataset(Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

        self.mean = torch.tensor(
            [0.485, 0.456, 0.406]
        ).view(3,1,1)

        self.std = torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(3,1,1)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, label = self.dataset[index]
        image = (image-self.mean) / self.std
        return image, label


adaptation_dataset = NormalizeDataset(
    adaptation_dataset
)

evaluation_dataset = NormalizeDataset(
    evaluation_dataset
)

In [ ]:
class UnlabeledDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):

